# DBSCAN Clustering Exercise

## What is DBSCAN?

**DBSCAN** (Density-Based Spatial Clustering of Applications with Noise) is a clustering algorithm that groups points based on how densely they are packed together.

It is different from K-Means in three important ways:

| Feature | K-Means | DBSCAN |
|---------|---------|--------|
| Number of clusters | You must specify k | Discovered automatically |
| Cluster shape | Round blobs only | Any shape |
| Outliers | Forced into a cluster | Labelled as **noise (−1)** |

## The Two Key Parameters

```
eps          → the search radius around each point ("how far to look for neighbours")
min_samples  → minimum neighbours needed within eps to form a dense region
```

Every point is classified as one of:
- **Core point** — has at least `min_samples` neighbours within `eps`
- **Border point** — fewer than `min_samples` neighbours, but within `eps` of a core point
- **Noise point** — not near any core point → labelled **−1** by sklearn

---

## Our Scenario

You work for a coffee shop chain. The loyalty team has collected data on customers:
- `visits_per_week` — how many times a customer visits each week
- `avg_spend_gbp` — their average spend per visit (£)

Your task is to use DBSCAN to segment customers into natural groups and identify unusual customers who don't fit any group.

## Steps

| Step | Task |
|------|------|
| 1 | Import libraries and create the dataset |
| 2 | Explore and plot the raw data |
| 3 | Scale the features |
| 4 | Find the best eps using the k-distance plot |
| 5 | Fit DBSCAN and inspect the labels |
| 6 | Visualise, interpret and compare with K-Means |

**Complete every `# YOUR CODE HERE` section before moving to the next step.**

Companion file: `DBSCAN_Exercise_Solutions.ipynb`

## Step 1 — Import Libraries and Create the Dataset

### Key Concept: The Libraries We Need

| Library | Purpose |
|---------|----------|
| `numpy` | Numerical arrays and random number generation |
| `pandas` | DataFrames — labelled tabular data |
| `matplotlib.pyplot` | Drawing charts |
| `DBSCAN` | The clustering algorithm |
| `KMeans` | For comparison in Step 6 |
| `StandardScaler` | Scales features to the same range before clustering |
| `NearestNeighbors` | Used to compute distances for the k-distance plot |

### What to do
1. Import all the libraries shown in the guide below
2. Run the dataset creation cell (do not modify it)
3. Print the shape of `df` and display the first 5 rows
4. Print summary statistics with `.describe()`
5. Check for missing values with `.isnull().sum()`

```python
# Import guide:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN, KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
```

In [ ]:
# YOUR CODE HERE — import the libraries listed in the guide above


import warnings
warnings.filterwarnings('ignore')
print("Libraries imported.")

In [ ]:
# ── Dataset creation — DO NOT MODIFY THIS CELL ───────────────────────
# Three natural customer groups + scattered outliers

rng = np.random.default_rng(42)

# Group 1 — Occasional customers: visit rarely, spend little
occasional = rng.normal(loc=[1.5, 3.5],  scale=[0.3, 0.4], size=(60, 2))

# Group 2 — Regular customers: visit often, spend moderately
regular    = rng.normal(loc=[4.0, 6.5],  scale=[0.4, 0.5], size=(70, 2))

# Group 3 — Loyal big-spenders: visit very often, spend a lot
loyal      = rng.normal(loc=[6.5, 11.0], scale=[0.4, 0.6], size=(50, 2))

# Outliers — unusual customers scattered across the space
outliers   = rng.uniform(low=[0, 0], high=[8, 14], size=(20, 2))

# Stack all groups and build a DataFrame
data = np.vstack([occasional, regular, loyal, outliers])
df   = pd.DataFrame(data.clip(min=0), columns=['visits_per_week', 'avg_spend_gbp'])

print(f"Dataset: {df.shape[0]} customers, {df.shape[1]} features")

In [ ]:
# 3. Print shape and display first 5 rows
# YOUR CODE HERE


In [ ]:
# 4. Summary statistics
# YOUR CODE HERE


In [ ]:
# 5. Check for missing values
# YOUR CODE HERE


**Reflection:** How many customers are in the dataset? What are the approximate ranges of `visits_per_week` and `avg_spend_gbp`? Are there any missing values?

*Your answer:*

---
## Step 2 — Explore and Plot the Raw Data

### Key Concept: Always Look Before You Cluster

Plotting the data **before** clustering lets you:
- See if natural groups are visible
- Spot potential outliers
- Check that both features are on reasonable scales

At this stage we have no labels — every dot is the same colour. This is the unsupervised starting point.

### What to do
Create a scatter plot of the raw unlabelled data:
- `visits_per_week` on the x-axis
- `avg_spend_gbp` on the y-axis
- A clear title and axis labels

```python
# Scatter plot template:
plt.figure(figsize=(8, 5))
plt.scatter(df['column_x'], df['column_y'],
            color='steelblue', alpha=0.6, edgecolors='white', s=60)
plt.xlabel('X Axis Label', fontsize=12)
plt.ylabel('Y Axis Label', fontsize=12)
plt.title('Your Title Here', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
```

In [ ]:
# YOUR CODE HERE — scatter plot of the raw unlabelled data


**Reflection:** How many natural groups can you see in the chart? Can you already spot any points that look like outliers (isolated from the main groups)?

*Your answer:*

---
## Step 3 — Scale the Features

### Key Concept: Why Scaling Matters for DBSCAN

DBSCAN finds neighbours by measuring **distance** between points. If one feature uses larger numbers than another, it will dominate the distance calculation and bias the clustering.

**Example from our data:**
- `visits_per_week` ranges roughly 0–8
- `avg_spend_gbp` ranges roughly 0–14

A £1 difference in spend would look larger than a 1-visit difference in visits, even though both may be equally important. `StandardScaler` solves this by converting every feature to:

```
mean = 0,   standard deviation = 1

scaled_value = (original_value − mean) / std
```

### What to do
1. Create a `StandardScaler` object
2. Fit and transform the two columns of `df` — store the result as `X_scaled`
3. Print the mean and standard deviation of both columns after scaling to confirm they are ~0 and ~1

```python
# Scaling template:
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(df[['column1', 'column2']])

# Check the result:
print(f"Mean after scaling : {X_scaled.mean(axis=0).round(3)}")
print(f"Std  after scaling : {X_scaled.std(axis=0).round(3)}")
```

In [ ]:
# 1 & 2. Create scaler and fit-transform df
# YOUR CODE HERE
scaler   = ...
X_scaled = ...

# 3. Print before and after comparison
print("Before scaling:")
print(f"  visits_per_week  — mean: {df['visits_per_week'].mean():.2f},  std: {df['visits_per_week'].std():.2f}")
print(f"  avg_spend_gbp    — mean: {df['avg_spend_gbp'].mean():.2f},  std: {df['avg_spend_gbp'].std():.2f}")

print("\nAfter scaling:")
# YOUR CODE HERE — print mean and std of X_scaled


**Reflection:** After scaling, what are the mean and standard deviation of each feature? Why is it important that both features now have the same scale before running DBSCAN?

*Your answer:*

---
## Step 4 — Find the Best eps Using the k-Distance Plot

### Key Concept: Choosing the Two Parameters

**`min_samples`** — controls how many neighbours a point needs to be considered a core point. A good starting rule: use **2 × number of features**. With 2 features → `min_samples = 5`.

**`eps`** — the search radius. Too small and everything becomes noise. Too large and all points merge into one cluster. The **k-distance plot** helps you find the right value:

```
1. For each point, compute its distance to its k-th nearest neighbour
   (where k = min_samples)
2. Sort these distances from smallest to largest and plot them
3. Look for the ELBOW — the point where the curve bends sharply upward
4. The distance at the elbow is your eps
```

**Why the elbow?** Points to the left are in dense regions (short distances). Points to the right are in sparse regions (long distances). The elbow is where density transitions from dense to sparse — exactly the boundary DBSCAN should use.

### What to do
1. Set `k = 5` (matching `min_samples`)
2. Fit a `NearestNeighbors` model to `X_scaled` and compute distances
3. Extract the k-th neighbour distance for every point and sort it
4. The visualisation code is provided — run it to see the elbow
5. Print your chosen `eps` value based on the elbow

```python
# Step-by-step guide:
k  = 5
nn = NearestNeighbors(n_neighbors=k)
nn.fit(X_scaled)                              # fit on scaled data
distances, _ = nn.kneighbors(X_scaled)        # get distances to k neighbours
kth_distances = np.sort(distances[:, k-1])    # take the k-th column, sort ascending
```

In [ ]:
# 1, 2 & 3. Compute k-th nearest neighbour distances
k = 5

# YOUR CODE HERE — create NearestNeighbors, fit, compute distances
nn = ...


# Sort the k-th neighbour distances
kth_distances = ...  # YOUR CODE HERE

print(f"Computed {len(kth_distances)} distances.")
print(f"Smallest 5 distances: {kth_distances[:5].round(3)}")
print(f"Largest  5 distances: {kth_distances[-5:].round(3)}")

In [ ]:
# 4. Plot the k-distance graph — visualisation code provided
# Look for the sharp bend (elbow) in the curve
# The distance at the elbow is a good choice for eps

plt.figure(figsize=(8, 5))

# The sorted distances
plt.plot(kth_distances, color='steelblue', linewidth=2, label='k-th distance')

# Mark your chosen eps with a horizontal dashed red line
# Change the y=... value below to match the elbow you see in the curve
chosen_eps = 0.4   # <── adjust this after looking at the plot
plt.axhline(y=chosen_eps, color='red', linestyle='--', linewidth=1.5,
            label=f'Chosen eps = {chosen_eps}')

plt.xlabel('Points (sorted by distance)', fontsize=12)
plt.ylabel(f'Distance to {k}th Nearest Neighbour', fontsize=12)
plt.title('k-Distance Plot — Finding Optimal eps', fontsize=13, fontweight='bold')
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

# 5. Print your chosen eps
print(f"Chosen eps = {chosen_eps}")
print("Adjust chosen_eps above if the red line does not sit at the elbow.")

**Reflection:** Where does the elbow appear in the curve? What does the flat section to the left of the elbow represent? What does the steep rise to the right represent?

*Your answer:*

---
## Step 5 — Fit DBSCAN and Inspect the Labels

### Key Concept: Fitting and Reading the Output

Fitting DBSCAN is a single line. The output is an array of **integer labels** — one per data point:

```
0, 1, 2 …  → cluster numbers (DBSCAN decides these automatically)
-1         → noise / outlier (not assigned to any cluster)
```

To understand the result, you need to count:
- How many distinct clusters were found
- How many points were labelled as noise

```python
# Fitting template:
dbscan = DBSCAN(eps=chosen_eps, min_samples=5)
labels = dbscan.fit_predict(X_scaled)

# Count clusters (exclude -1 which is noise)
n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
n_noise    = (labels == -1).sum()
```

### What to do
1. Fit DBSCAN using your `chosen_eps` from Step 4 and `min_samples=5`
2. Print the number of clusters found and number of noise points
3. Loop through the unique labels and print how many points belong to each

In [ ]:
# 1. Fit DBSCAN
# YOUR CODE HERE
dbscan = ...
labels = ...


# 2. Count clusters and noise
# YOUR CODE HERE
n_clusters = ...
n_noise    = ...

print(f"Clusters found : {n_clusters}")
print(f"Noise points   : {n_noise}")

In [ ]:
# 3. Print point counts per label
print("Label breakdown:")
for label in sorted(set(labels)):
    # YOUR CODE HERE
    # Hint: name = 'Noise' if label == -1 else f'Cluster {label}'
    # count = (labels == label).sum()
    pass

**Reflection:** How many clusters did DBSCAN find? Does this match the number of natural groups you spotted in Step 2? How many noise points were found — does that seem reasonable given the outliers in the data?

*Your answer:*

---
## Step 6 - Visualise, Interpret and Compare with K-Means

### Key Concept: Three Things in This Final Step

**Part A — Plot the DBSCAN clusters.**  
Colour each cluster differently. Plot noise points as grey crosses so they are immediately distinguishable.

**Part B — Summarise and name each cluster.**  
Compute the mean feature values per cluster to understand what each group represents, then give each a descriptive label.

**Part C — Compare with K-Means side by side.**  
Run K-Means with the same number of clusters and plot both results next to each other. Notice how K-Means handles the outliers differently.

### What to do
- For **Part A**: fill in the two `plt.scatter()` calls
- For **Part B**: fill in the `groupby` summary and the `labels_map` dictionary
- For **Part C**: fill in the `KMeans` call — visualisation is provided

### Part A — Plot the DBSCAN Clusters

```python
# Pattern for plotting clusters and noise:

df['cluster'] = labels     # attach labels to DataFrame for easy filtering

cluster_colours = {0: '#e74c3c', 1: '#3498db', 2: '#2ecc71', 3: '#f39c12'}

for cluster_id, colour in cluster_colours.items():
    mask = df['cluster'] == cluster_id    # select only this cluster's rows
    if mask.sum() > 0:                    # skip if cluster doesn't exist
        plt.scatter(df.loc[mask, 'col_x'], df.loc[mask, 'col_y'],
                    c=colour, label=f'Cluster {cluster_id}',
                    s=70, alpha=0.8, edgecolors='white')

# Noise points — grey X markers
noise_mask = df['cluster'] == -1
plt.scatter(df.loc[noise_mask, 'col_x'], df.loc[noise_mask, 'col_y'],
            c='grey', marker='x', s=80, linewidths=1.5,
            label=f'Noise ({noise_mask.sum()} points)')
```

In [ ]:
# Attach labels to the DataFrame
df['cluster'] = labels

cluster_colours = {0: '#e74c3c', 1: '#3498db', 2: '#2ecc71', 3: '#f39c12'}

plt.figure(figsize=(8, 6))

# Plot each cluster
for cluster_id, colour in cluster_colours.items():
    mask = df['cluster'] == cluster_id
    if mask.sum() > 0:
        # YOUR CODE HERE — plt.scatter() for this cluster's points
        pass

# Plot noise points
noise_mask = df['cluster'] == -1
# YOUR CODE HERE — plt.scatter() for noise points (grey, marker='x')


plt.xlabel('Visits per Week', fontsize=12)
plt.ylabel('Avg Spend (£)', fontsize=12)
plt.title('DBSCAN — Coffee Shop Customer Segments', fontsize=13, fontweight='bold')
plt.legend(fontsize=10)
plt.tight_layout()
plt.show()

### Part B — Summarise and Label Each Cluster

```python
# Group by cluster, compute mean of each feature, count members:
summary = (
    df[df['cluster'] != -1]             # exclude noise
    .groupby('cluster')[['col1', 'col2']]
    .mean()
    .round(2)
)
summary['count'] = df[df['cluster'] != -1].groupby('cluster')['cluster'].count()

# Map cluster numbers to meaningful names:
labels_map = {0: 'Group Name A', 1: 'Group Name B', 2: 'Group Name C'}
summary['customer_type'] = summary.index.map(labels_map)
```

In [ ]:
# Compute cluster summary
# YOUR CODE HERE
summary = ...
summary['count'] = ...

print("Cluster Profiles (noise excluded):")
print(summary)

In [ ]:
# Assign descriptive labels — look at the summary above to decide which cluster is which
# YOUR CODE HERE — edit the dictionary values to match your summary
labels_map = {
    0: '???',     # low visits, low spend?
    1: '???',     # moderate visits, moderate spend?
    2: '???',     # high visits, high spend?
}

summary['customer_type'] = summary.index.map(labels_map)
print(summary[['customer_type', 'visits_per_week', 'avg_spend_gbp', 'count']])

### Part C — Compare DBSCAN with K-Means

Run K-Means with the same number of clusters that DBSCAN found. The visualisation code is fully provided — just fill in the K-Means fit line.

In [ ]:
# Fit K-Means with the same number of clusters DBSCAN found
# YOUR CODE HERE
kmeans        = KMeans(n_clusters=n_clusters, init='k-means++', n_init=10, random_state=42)
kmeans_labels = ...   # call .fit_predict(X_scaled)

In [ ]:
# Side-by-side comparison — visualisation code provided
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Left plot: K-Means ────────────────────────────────────────────────
axes[0].scatter(df['visits_per_week'], df['avg_spend_gbp'],
                c=kmeans_labels, cmap='viridis',
                s=60, alpha=0.7, edgecolors='white')
axes[0].set_title(f'K-Means (k={n_clusters})\nEvery point forced into a cluster',
                  fontsize=12, fontweight='bold')
axes[0].set_xlabel('Visits per Week')
axes[0].set_ylabel('Avg Spend (£)')

# ── Right plot: DBSCAN ────────────────────────────────────────────────
colour_map = {0: '#e74c3c', 1: '#3498db', 2: '#2ecc71', -1: 'grey'}

for lbl in sorted(set(labels)):
    mask  = df['cluster'] == lbl
    name  = 'Noise' if lbl == -1 else f'Cluster {lbl}'
    mrk   = 'x'     if lbl == -1 else 'o'
    axes[1].scatter(df.loc[mask, 'visits_per_week'],
                    df.loc[mask, 'avg_spend_gbp'],
                    c=colour_map.get(lbl, 'black'),
                    marker=mrk, s=60, alpha=0.8,
                    edgecolors='white', label=name)

axes[1].set_title(f'DBSCAN (eps={chosen_eps}, min_samples=5)\nOutliers detected as noise',
                  fontsize=12, fontweight='bold')
axes[1].set_xlabel('Visits per Week')
axes[1].set_ylabel('Avg Spend (£)')
axes[1].legend(fontsize=9)

plt.suptitle('K-Means vs DBSCAN — Same Data, Different Approaches',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("K-Means : outliers are silently absorbed into whichever cluster is nearest.")
print("DBSCAN  : outliers are correctly identified and flagged as noise.")

**Final Reflection — answer all three questions:**

1. Looking at the side-by-side chart, how did K-Means handle the outlier points compared to DBSCAN? Which result looks more realistic for a business scenario?

2. For each customer segment DBSCAN found, suggest one specific marketing action the coffee shop could take:

| Segment | Suggested Marketing Action |
|---------|----------------------------|
| ??? | ??? |
| ??? | ??? |
| ??? | ??? |

3. What might happen if you set `eps` much too large (e.g. `eps=5.0`)? What if it is too small (e.g. `eps=0.05`)?

*Your answers:*

1. 

2. 

3. 